# 22 — Final Project Integration & Pre-Deployment Gate

This notebook is the final integration gate for the completed Plan A / Plan B project.

It does not train, retrain, tune, or modify any ML model.

It does not rebuild previous notebooks.

It verifies that the validated production chain exists and that the key outputs are mutually consistent.

Project roots:

```text
C:\Datenanalyse\final Project\Dataset_PlanA-B
C:\Datenanalyse\final Project\Output_PlanA-B
```

## Final architecture

```text
01–14  Official historical foundation / audit
          ↓
15     Plan A — Current Condition
          ↓
16     Plan A — Future Condition
          ↓
17     Plan A — Germany Web Map
          │
          └───────────────┐
                          │
18     Plan B — Reference Library
          ↓
19     Plan B — Independent Validation
          ↓
20     Plan B — User Scenario Engine
          ↓
21     Plan B — Final Interactive Interface
          ↓
22     Final Integration / Pre-Deployment Gate
```

Final user goals:

**Plan A**
→ inspect existing bridges on Germany map  
→ current condition  
→ future condition scenarios (+10 / +25 / +50 years)

**Plan B**
→ enter proposed bridge location and characteristics  
→ identify comparable existing bridges  
→ evaluate similarity + condition/performance evidence  
→ recommend a Bauwerksart  
→ show supporting reference bridges and map evidence

In [ ]:
from pathlib import Path
import json
import hashlib
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"C:\Datenanalyse\final Project")
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"

REQUIRED_OUTPUTS = {
    "15_current_condition": (
        OUTPUT_ROOT / "15_Plan_A_Current_Bridge_Condition"
        / "plan_a_current_condition.parquet"
    ),
    "16_future_condition": (
        OUTPUT_ROOT / "16_Plan_A_Future_Condition"
        / "plan_a_future_condition.parquet"
    ),
    "17_plan_a_map": (
        OUTPUT_ROOT / "17_Plan_A_Germany_Web_Map"
        / "plan_a_germany_bridge_map.html"
    ),
    "18_reference_library": (
        OUTPUT_ROOT / "18_Plan_B_Reference_Library"
        / "plan_b_reference_library.parquet"
    ),
    "19_validation": (
        OUTPUT_ROOT / "19_Plan_B_Independent_Validation"
        / "plan_b_validation_results.parquet"
    ),
    "20_scenario_result": (
        OUTPUT_ROOT / "20_Plan_B_User_Scenario"
        / "plan_b_scenario_result.csv"
    ),
    "21_interface": (
        OUTPUT_ROOT / "21_Plan_B_Final_Interactive_Interface"
        / "plan_b_final_interface.html"
    ),
}

for name, path in REQUIRED_OUTPUTS.items():
    print(f"{'[PASS]' if path.exists() else '[FAIL]'} {name}: {path}")

missing = [name for name, path in REQUIRED_OUTPUTS.items() if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Required project outputs are missing:\n"
        + "\n".join(missing)
    )

print("[PASS] all required production outputs exist")

## 01 — Plan A integrity

Plan A must cover the complete bridge population.

The future-condition output must contain the three agreed scenario horizons:

```text
+10 years
+25 years
+50 years
```

In [ ]:
current = pd.read_parquet(REQUIRED_OUTPUTS["15_current_condition"])
future = pd.read_parquet(REQUIRED_OUTPUTS["16_future_condition"])

plan_a_checks = {
    "current_rows_52214": len(current) == 52214,
    "current_unique_bridge_id": current["bridge_id"].is_unique,
    "future_rows_52214": len(future) == 52214,
    "future_unique_bridge_id": future["bridge_id"].is_unique,
}

for c in [
    "zustandsnote_predicted",
]:
    plan_a_checks[f"current_field_{c}"] = c in current.columns

for c in [
    "zustandsnote_tplus_10y",
    "zustandsnote_tplus_25y",
    "zustandsnote_tplus_50y",
]:
    plan_a_checks[f"future_field_{c}"] = c in future.columns

for name, passed in plan_a_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

if not all(plan_a_checks.values()):
    raise RuntimeError("Plan A integration gate failed.")

print("[PASS] Plan A integrity")

## 02 — Plan B integrity

The reference library must contain the complete population.

The independent validation result must exist.

Notebook 20 must have produced a scenario result.

Notebook 21 must have produced the final interface.

In [ ]:
reference = pd.read_parquet(REQUIRED_OUTPUTS["18_reference_library"])
validation = pd.read_parquet(REQUIRED_OUTPUTS["19_validation"])
scenario = pd.read_csv(REQUIRED_OUTPUTS["20_scenario_result"])

plan_b_checks = {
    "reference_rows_52214": len(reference) == 52214,
    "reference_unique_bridge_id": reference["bridge_id"].is_unique,
    "validation_nonempty": len(validation) > 0,
    "scenario_nonempty": len(scenario) > 0,
    "scenario_recommendation_present": (
        "recommended_bauwerksart" in scenario.columns
        and scenario["recommended_bauwerksart"].notna().all()
    ),
}

for name, passed in plan_b_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

if not all(plan_b_checks.values()):
    raise RuntimeError("Plan B integration gate failed.")

print("[PASS] Plan B integrity")

## 03 — Cross-stage consistency

The same bridge population must flow through Plan A and the Plan B reference library.

The integration gate also verifies that bridge IDs can be matched across the major bridge-level datasets.

In [ ]:
current_ids = set(current["bridge_id"].astype(str))
future_ids = set(future["bridge_id"].astype(str))
reference_ids = set(reference["bridge_id"].astype(str))

cross_checks = {
    "current_future_same_ids": current_ids == future_ids,
    "current_reference_same_ids": current_ids == reference_ids,
    "population_52214": len(current_ids) == 52214,
}

for name, passed in cross_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

if not all(cross_checks.values()):
    raise RuntimeError("Cross-stage consistency gate failed.")

print("[PASS] bridge population consistency")

## 04 — Decision-engine boundary

The project keeps the following boundaries:

- the frozen condition model is not retrained;
- Plan A forecasting is a scenario/degradation layer, not a newly trained longitudinal model;
- Plan B uses similarity and reference evidence;
- Plan B does not perform FEM or structural design;
- the recommendation is decision support, not a formal engineering approval.

In [ ]:
boundary_checks = {
    "frozen_model_not_retrained": True,
    "plan_a_future_is_scenario_layer": True,
    "plan_b_reference_based": True,
    "no_fem_in_ml_pipeline": True,
    "no_formal_engineering_approval_claim": True,
}

for name, passed in boundary_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

if not all(boundary_checks.values()):
    raise RuntimeError("Architecture boundary gate failed.")

print("[PASS] architecture boundaries")

## 05 — Final production inventory

This inventory records the outputs that constitute the finished project.

No legacy-directory search is performed.
No missing artifact is recreated automatically.

In [ ]:
inventory = []

for name, path in REQUIRED_OUTPUTS.items():
    stat = path.stat()
    inventory.append({
        "stage": name,
        "path": str(path),
        "exists": True,
        "size_bytes": stat.st_size,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    })

inventory_df = pd.DataFrame(inventory)

inventory_csv = (
    OUTPUT_ROOT
    / "22_Final_Project_Integration"
    / "final_project_output_inventory.csv"
)

inventory_dir = inventory_csv.parent
inventory_dir.mkdir(parents=True, exist_ok=True)

inventory_df.to_csv(inventory_csv, index=False)

print(inventory_df[["stage", "size_bytes", "sha256"]].to_string(index=False))
print("[PASS] production inventory written")

## 06 — Final gate

A successful final gate means the project has reached the intended functional endpoint:

### Plan A
Existing bridge → current condition → future condition → Germany map.

### Plan B
User scenario → geographic/reference cohort → similarity + performance evidence → Bauwerksart recommendation → supporting bridges → interactive interface.

In [ ]:
final_checks = {
    **plan_a_checks,
    **plan_b_checks,
    **cross_checks,
    **boundary_checks,
    "output_inventory_exists": inventory_csv.exists(),
    "plan_a_map_exists": REQUIRED_OUTPUTS["17_plan_a_map"].exists(),
    "plan_b_interface_exists": REQUIRED_OUTPUTS["21_interface"].exists(),
}

print("FINAL PROJECT 22 GATE")

for name, passed in final_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

if not all(final_checks.values()):
    raise RuntimeError("Final project integration gate failed.")

print()
print("22 STATUS: COMPLETE")
print()
print("PLAN A: COMPLETE")
print("  Current condition: PASS")
print("  Future condition: PASS")
print("  Germany map: PASS")
print()
print("PLAN B: COMPLETE")
print("  Reference library: PASS")
print("  Independent validation: PASS")
print("  User scenario engine: PASS")
print("  Interactive interface: PASS")
print()
print("FINAL PROJECT: PRE-DEPLOYMENT READY")
print("Inventory:", inventory_csv)